# ConvNeXt OOF logit-residual screen
Fixed private residual-distillation screen; organizer gold is evaluation-only; no submission.


In [ ]:
from pathlib import Path
import gc, hashlib, json, os, random, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.random_projection import SparseRandomProjection
from sklearn.linear_model import Ridge
from torch.utils.data import DataLoader, TensorDataset

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]
SEEDS = (20261031, 20261101, 20261102, 20261103, 20261104)
FEATURE_SHA = "7f1986bb30cf92dbbf440b43a5a5933c1a3d13a0bb61d91950457cb0f8e018cc"
CHECKPOINT_SHA = "0b64baa9d1c29c8d5c4dffe06d9ab2c941119b37738e5a2346e882ab694e7b80"
KNEE_SHA = "314c2af387a5cbd1d25f192f19f4ecc6d4ae4b30403e92e9d9b3ed2c325880b1"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ANCHOR_SHA = "6cdea905549c3df74d94c08a0547a556c9b3a890f6b1d90e432f0b79d3b0c124"
RESIDUAL_WEIGHT = .10
RIDGE_ALPHA = 100.0
PROJECTION_DIM = 384
BAG_FRACTION = .85

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def unique_input(name):
    hits = list(Path("/kaggle/input").rglob(name))
    if len(hits) != 1:
        raise RuntimeError(f"expected one {name}, found {len(hits)}: {hits[:8]}")
    return hits[0]

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def macro_auc(y, p):
    scores = []
    for j in range(y.shape[1]):
        if len(np.unique(y[:, j])) > 1:
            scores.append(roc_auc_score(y[:, j], p[:, j]))
    return float(np.mean(scores))

def paired_bootstrap(y, control, candidate, draws=5000, seed=20261105):
    rng = np.random.default_rng(seed); delta = []
    for _ in range(draws):
        ix = rng.integers(0, len(y), len(y))
        delta.append(macro_auc(y[ix], candidate[ix]) - macro_auc(y[ix], control[ix]))
    delta = np.asarray(delta)
    return {"mean": float(delta.mean()),
            "p_improve": float((delta > 0).mean()),
            "q025": float(np.quantile(delta, .025)),
            "q975": float(np.quantile(delta, .975))}

if DEVICE.type != "cuda":
    raise RuntimeError("GPU required")
feature_path = unique_input("convnext_reader_fold0_features.npz")
if feature_path.stat().st_size != 363928367 or sha256(feature_path) != FEATURE_SHA:
    raise RuntimeError("feature artifact identity mismatch")
checkpoint_path = unique_input("cnxt_v0_fold0.pt")
if sha256(checkpoint_path) != CHECKPOINT_SHA:
    raise RuntimeError("checkpoint identity mismatch")
knee_path = unique_input("knee.py")
if sha256(knee_path) != KNEE_SHA:
    raise RuntimeError("model source identity mismatch")
sys.path.insert(0, str(knee_path.parent))
from knee import KneeNet

train_path = unique_input("train.csv")
train = pd.read_csv(train_path, dtype={"StudyInstanceUID": str})
ids = train.StudyInstanceUID.astype(str).to_numpy()
gold = train[TARGETS].notna().all(axis=1).to_numpy()
if len(ids) != 4407 or len(set(ids)) != 4407 or int(gold.sum()) != 58:
    raise RuntimeError("competition identity/gold drift")

package = np.load(feature_path, allow_pickle=False)
if package["ids"].astype(str).tolist() != ids.tolist():
    raise RuntimeError("feature study ordering mismatch")
features = package["features"]
masks = package["masks"]
positions = package["positions"]
if features.shape != (4407, 6, 12, 768) or features.dtype != np.float16:
    raise RuntimeError("feature shape/dtype drift")
if masks.shape != (4407, 6) or not np.isin(masks, [0, 1]).all():
    raise RuntimeError("mask drift")
if positions.shape != (4407, 6, 12) or not np.isfinite(positions).all():
    raise RuntimeError("position drift")

label_path = unique_input("llm_labels_v4_blend.csv")
label_frame = pd.read_csv(label_path, dtype={"StudyInstanceUID": str})
aligned = train[["StudyInstanceUID"]].merge(
    label_frame, on="StudyInstanceUID", how="left", validate="one_to_one")
steven = aligned[TARGETS].to_numpy(np.float32)
if not np.isfinite(steven).all():
    raise RuntimeError("Steven labels incomplete")

oof = pd.read_csv(unique_input("convnext_oof_predictions.csv"),
                  dtype={"StudyInstanceUID": str})
if oof.columns.tolist() != ["StudyInstanceUID", "teacher_fold"] + TARGETS:
    raise RuntimeError("OOF schema drift")
if len(oof) != 2610 or oof.StudyInstanceUID.nunique() != 2610:
    raise RuntimeError("OOF coverage drift")
if set(oof.teacher_fold.unique()) != {0, 1, 2}:
    raise RuntimeError("OOF fold drift")
if set(oof.StudyInstanceUID) & set(train.loc[gold, "StudyInstanceUID"]):
    raise RuntimeError("gold leakage in teacher table")
teacher = train[["StudyInstanceUID"]].merge(
    oof, on="StudyInstanceUID", how="left", validate="one_to_one")
covered = teacher[TARGETS].notna().all(axis=1).to_numpy()
covered_idx = np.flatnonzero(covered & ~gold)
fit_idx = np.flatnonzero(~gold)
if len(covered_idx) != 2610 or len(fit_idx) != 4349 or covered[gold].any():
    raise RuntimeError("teacher alignment failure")
corrected = steven.copy()
teacher_values = teacher[TARGETS].to_numpy(np.float32)
corrected[covered_idx] = .5 * steven[covered_idx] + .5 * teacher_values[covered_idx]


def ranks(p):
    out = np.empty_like(p, dtype=np.float64)
    for j in range(p.shape[1]):
        order = np.argsort(p[:, j], kind="mergesort")
        values = p[order, j]
        ranked = np.empty(len(p), np.float64)
        start = 0
        while start < len(p):
            stop = start + 1
            while stop < len(p) and values[stop] == values[start]: stop += 1
            ranked[order[start:stop]] = .5 * (start + stop - 1)
            start = stop
        out[:, j] = (ranked + .5) / len(p)
    return out

def target_aucs(y, p):
    return np.asarray([roc_auc_score(y[:, j], p[:, j]) for j in range(12)])

def logit(p):
    p = np.clip(p, .02, .98)
    return np.log(p / (1. - p))

def sigmoid(x):
    return 1. / (1. + np.exp(-x))

def head_predict(model, indices):
    model.eval(); rows = []
    with torch.no_grad():
        for start in range(0, len(indices), 128):
            ix = indices[start:start + 128]
            x = torch.from_numpy(features[ix]).to(DEVICE)
            m = torch.from_numpy(masks[ix]).to(DEVICE)
            p = torch.from_numpy(positions[ix]).to(DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                rows.append(torch.sigmoid(model.head(x, m.bool(), p)).float().cpu().numpy())
    return np.concatenate(rows)

# Exact untouched public reader predictions.
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
public = KneeNet(checkpoint["args"]["backbone"].split(".")[0], pretrained=False)
public.load_state_dict(checkpoint["model"], strict=True); del public.enc
public = public.to(DEVICE).eval()
all_idx = np.arange(len(ids)); gold_idx = np.flatnonzero(gold)
public_prediction = head_predict(public, all_idx)
del public; torch.cuda.empty_cache()

# Fixed label-free sparse projection.
f = features.astype(np.float32)
slot_mean = f.mean(axis=2)
slot_std = f.std(axis=2)
representation = np.concatenate([slot_mean, slot_std], axis=2)
representation *= masks[:, :, None].astype(np.float32)
representation = representation.reshape(len(ids), -1)
projector = SparseRandomProjection(n_components=PROJECTION_DIM,
                                   dense_output=True, random_state=20261018)
z_fit = projector.fit_transform(representation[fit_idx]).astype(np.float32)
z_gold = projector.transform(representation[gold_idx]).astype(np.float32)
mean = z_fit.mean(0); std = np.maximum(z_fit.std(0), 1e-5)
z_fit = (z_fit - mean) / std; z_gold = (z_gold - mean) / std

covered_fit = np.flatnonzero(np.isin(fit_idx, covered_idx))
if len(covered_fit) != 2610: raise RuntimeError("covered fit alignment drift")
residual_target = np.clip(logit(corrected[fit_idx][covered_fit]) -
                          logit(steven[fit_idx][covered_fit]), -2., 2.)

# Immutable four-reader anchor.
anchor_path = unique_input("multireader_gold_predictions.npz")
if sha256(anchor_path) != ANCHOR_SHA: raise RuntimeError("anchor hash drift")
anchor_pkg = np.load(anchor_path, allow_pickle=False)
if anchor_pkg["ids"].astype(str).tolist() != ids[gold].tolist():
    raise RuntimeError("anchor ID drift")
truth = anchor_pkg["truth"].astype(np.float64)
if not np.array_equal(truth, train.loc[gold, TARGETS].to_numpy(np.float64)):
    raise RuntimeError("anchor truth drift")
if anchor_pkg["arm_names"].astype(str).tolist() != ["r6_flip","raptor384","raptor224","r7"]:
    raise RuntimeError("anchor arm drift")
arm = anchor_pkg["predictions"].astype(np.float64)
anchor = ranks(np.einsum("nat,a->nt", arm, np.asarray([.50,.325,.175,0.])))
anchor_auc = macro_auc(truth, anchor)
public_gold = public_prediction[gold_idx]
public_auc = macro_auc(truth, public_gold)

seed_predictions = []; seed_rows = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    bag = np.sort(rng.choice(covered_fit, int(BAG_FRACTION * len(covered_fit)), replace=False))
    model = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True, solver="lsqr", tol=1e-5)
    model.fit(z_fit[bag], residual_target[bag])
    correction = np.clip(model.predict(z_gold), -1., 1.)
    corrected_gold = sigmoid(logit(public_gold) + correction)
    candidate = (1. - RESIDUAL_WEIGHT) * anchor + RESIDUAL_WEIGHT * ranks(corrected_gold)
    seed_predictions.append(corrected_gold.astype(np.float32))
    seed_rows.append({"seed": int(seed), "bag_size": int(len(bag)),
                      "reader_auc": macro_auc(truth, corrected_gold),
                      "blend_auc": macro_auc(truth, candidate),
                      "blend_minus_anchor": macro_auc(truth, candidate) - anchor_auc})
    print(seed_rows[-1], flush=True)

reader = np.mean(seed_predictions, axis=0)
candidate = (1. - RESIDUAL_WEIGHT) * anchor + RESIDUAL_WEIGHT * ranks(reader)
reader_auc = macro_auc(truth, reader); candidate_auc = macro_auc(truth, candidate)
target_delta = target_aucs(truth, candidate) - target_aucs(truth, anchor)
bootstrap = paired_bootstrap(truth, anchor, candidate, draws=10000, seed=20261018)
gate = {
    "reader_gain_at_least_0_002": reader_auc - public_auc >= .002,
    "anchor_gain_at_least_0_001": candidate_auc - anchor_auc >= .001,
    "nonnegative_targets_at_least_8": int((target_delta >= 0).sum()) >= 8,
    "positive_seeds_at_least_4": int(sum(r["blend_minus_anchor"] > 0 for r in seed_rows)) >= 4,
    "bootstrap_p_at_least_0_90": bootstrap["p_improve"] >= .90,
    "bootstrap_q025_above_minus_0_001": bootstrap["q025"] > -.001,
}
report = {
    "contract": "fixed OOF logit-residual on exact public ConvNeXt head; no submission",
    "feature_sha256": FEATURE_SHA, "checkpoint_sha256": CHECKPOINT_SHA,
    "source_knee_sha256": KNEE_SHA, "anchor_sha256": ANCHOR_SHA,
    "fit_studies": 4349, "corrected_studies": 2610, "unchanged_studies": 1739,
    "gold_fit_count": 0, "gold_evaluation_count": 58,
    "projection": "SparseRandomProjection(seed=20261018)",
    "projection_dim": PROJECTION_DIM, "ridge_alpha": RIDGE_ALPHA,
    "bag_fraction": BAG_FRACTION,
    "residual_clip": [-1., 1.], "residual_weight": RESIDUAL_WEIGHT,
    "seed_results": seed_rows,
    "ensemble": {"public_head_auc": public_auc, "corrected_reader_auc": reader_auc,
                 "reader_gain": reader_auc-public_auc, "anchor_auc": anchor_auc,
                 "candidate_auc": candidate_auc, "candidate_gain": candidate_auc-anchor_auc,
                 "nonnegative_targets": int((target_delta >= 0).sum())},
    "bootstrap": bootstrap, "gate": gate, "passed": bool(all(gate.values())),
    "submission_created": False,
}
out = Path("/kaggle/working")
np.savez_compressed(out / "convnext_oof_logit_residual_predictions.npz",
                    ids=ids[gold].astype("U"), truth=truth.astype(np.uint8),
                    public=public_gold.astype(np.float32), anchor=anchor.astype(np.float32),
                    reader=reader.astype(np.float32), candidate=candidate.astype(np.float32),
                    seeds=np.asarray(SEEDS), seed_predictions=np.asarray(seed_predictions))
(out / "convnext_oof_logit_residual_report.json").write_text(json.dumps(report, indent=2)+"\n")
for path in out.glob("*.csv"): path.unlink()
if list(out.glob("*.csv")): raise RuntimeError("forbidden CSV artifact")
print(json.dumps(report, indent=2), flush=True)
